## Compare rigid vs non-rigid z-motion correction
Test with T-series recorded with no calcium activity, only z motion.  
  
T-series data: `Scnn1aAi14_B1M0/04222024/TSeries-04222024-1047-001`  
Z-series: `Scnn1aAi14_B1M0/04222024/ZSeries-04222024-1047-001`  
   *Alternatively:  
   `C57_O1M1/09182023/TSeries-09182023-1503-003`  
   `C57_O1M1/09182023/ZSeries-09182023-1503-003`*

In [ ]:
from pathlib import Path
import json
import os
import numpy as np
import mesmerize_core as mc

import sys
sys.path.append("..")  # Add above scripts directory to python modules path.
import compute_zcorr as cz

import matplotlib.pyplot as plt

from tifffile import TiffFile, TiffWriter
from caiman.mmapping import load_memmap
import time

from scipy.ndimage import gaussian_filter

from tqdm import tqdm
from scipy.stats import mode, linregress
from scipy.ndimage import gaussian_filter, affine_transform, label
from skimage.transform import warp
from skimage.registration import phase_cross_correlation
from PIL import Image
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed

### Load paths, parameters

In [ ]:
path_file = '/home/wanglab/code/imaging_analysis/Analysis_2P/test/Paths_zmotion_lin2.json'

with open(path_file) as f:
    paths = json.load(f)
    
params_files = paths['params_files']
with open(params_files[0]) as f:
    parameters = json.load(f)

zstack_path = paths['zstack_paths']
zstack_path = Path(zstack_path[0])
z_params_files = paths['z_params_files']
with open(z_params_files[0]) as f:
    z_parameters = json.load(f)
    
z_shifted_file = z_parameters['zstack_shift']['file_name']

# Get the motion correction parameters
parameters_mcorr = parameters['params_mcorr']

### Run CaImAn’s motion correction	
Returns mcorr_movie path (memmap file)

In [ ]:
data_paths = paths['data_paths']
export_paths = paths['export_paths']
regex_pattern='*_Ch2_*.ome.tif'

export_path= Path(export_paths[0])
# If export path does not exist, create it
if not export_path.exists():
    os.makedirs(export_path)
# Set the parent raw data path
mc.set_parent_raw_data_path(export_path) 

# Run the x/y motion correction
# batch_path, index, mcorr_movie_path = pmc.run_mcorr(data_paths, export_path, parameters_mcorr, regex_pattern)

# Or set the path to the motion corrected file 
mcorr_movie_path = export_path / '88219db3-9755-4f7d-a8ca-740aebf51f84' / '88219db3-9755-4f7d-a8ca-740aebf51f84-cat_tiff_bt_els__d1_765_d2_765_d3_1_order_F_frames_1600.mmap'

# Concatenated tiff file(s) converted to uint16 range, from uint12 range, in pipeline_mcorr_cnmf > run_mcorr -> bruker_concat_tif > convert_to_bigtiff
# https://vscode.dev/github/pseudomanu/Analysis_2P/blob/dev_zcorr/Mesmerize/bruker_concat_tif.py#L195

### Shift z-stack, based on objective and stage angles
Function `shift_zstack` from `compute_zcorr`.  
Returns shifted_zstack (tiff file in Z-series folder)

In [ ]:
if not os.path.exists(zstack_path / z_shifted_file):
    shift_zstack_path=cz.shift_zstack(z_parameters['zstack_shift'], zstack_path, z_shifted_file)
else:
    shift_zstack_path=zstack_path / z_shifted_file

#  Shifted z-stack values converted to uint16 range, from uint12 range, in compute_zcorr > shift_zstack
#  https://vscode.dev/github/pseudomanu/Analysis_2P/blob/dev_zcorr/Mesmerize/compute_zcorr.py#L120

### Compute z-correlation: correlation and x/y shifts between mcorr_movie and the shifted z-stack
Function `compute_zcorrel` from `compute_zcorr`.
* Loads z-stack and flip it
* Computes shifts values  with `skimage.registration.phase_cross_correlation`. 
* Computes correlations with `numpy.corrcoef`.  
`zpos` is then computed as the best correlations after smoothing values.
* Returns the z-correlation between the T-series and the Z-series, with x and y shifts, and the estimated z position of the T-series' frames.  

In [ ]:
mesmerize_path = mcorr_movie_path.parents[1] # get mesmerize directory where the zcorr file will be saved
if not os.path.exists(mesmerize_path / "z_correlation.npz"):
    z_correlation=cz.compute_zcorrel(zstack_path / z_shifted_file, mcorr_movie_path) 
else:
    z_correlation=np.load(mesmerize_path / "z_correlation.npz")

In [ ]:
# Get zpos, x and y shifts
zpos = z_correlation['zpos']
xshift = z_correlation['xshift']
yshift = z_correlation['yshift']
zcorr = z_correlation['zcorr']

In [ ]:
# # Export zpos, xshift, yshift, zcorr as csv, no header, no index
# # pip install openpyxl
# zcorr_filepath = export_path / "zcorr_reg.xlsx"
# with pd.ExcelWriter(zcorr_filepath) as writer:
#     pd.DataFrame(zpos).to_excel(writer, sheet_name='zpos', index=False, header=False)
#     pd.DataFrame(xshift).to_excel(writer, sheet_name='xshift', index=False, header=False)
#     pd.DataFrame(yshift).to_excel(writer, sheet_name='yshift', index=False, header=False)
#     pd.DataFrame(zcorr).to_excel(writer, sheet_name='zcorr', index=False, header=False)


<div class="alert alert-info">
    <h3 style="margin-top: 0;">Compute a fit between F_anat_rigid and F_func for rigid motion correction.</h3>  

Load F_func (motion-corected movie). 

Compute F_anat_rigid: load Zstack, then find Zstack[:, :, zpos(frame)] using corresponding xshifts/yshifts.  

Linearize the two 3D arrays and find correlation and fit (b).  
</div>

**Create F_anat**

In [ ]:
# Load the motion-corrected movie
movie_16bit, dims, T = load_memmap(mcorr_movie_path)
F_func = np.reshape(movie_16bit.T, [T] + list(dims), order='F')
Nframe, Ny, Nx = F_func.shape
# Clip the movie range to uint16
F_func = np.clip(F_func, 0, 2**16-1)

# # Load the FOV file
# mcorr_batch_dir = mcorr_movie_path.parent
# fov_file_path = Path.joinpath(mcorr_batch_dir, f"{mcorr_batch_dir.stem}_mean_projection.npy")
# fov_image = np.load(fov_file_path)

# Load the shifted z-stack
info_zstack = Image.open(shift_zstack_path)
Nz = info_zstack.n_frames
Zstack = np.zeros((Ny, Nx, Nz), dtype=np.float32)
for iz in range(Nz):
    info_zstack.seek(iz)
    Zstack[:, :, iz] = np.array(info_zstack)
info_zstack.close()        
# TODO: check why on Windows F_anat bit depth is 8 and not 16
# Clip the z-stack range to uint16
Zstack = np.clip(Zstack, 0, 2**16-1)
# Flip Zstack up-down to match the orientation of the movie
Zstack = np.flip(Zstack, axis=0)

# Print F_func and Zstack shapes, data type and min/max range
print(f"F_func shape: {F_func.shape}, data type: {F_func.dtype}, min: {F_func.min()}, max: {F_func.max()}")
print(f"Zstack shape: {Zstack.shape}, data type: {Zstack.dtype}, min: {Zstack.min()}, max: {Zstack.max()}")

In [ ]:
# Plot the first image of the movie, the FOV image, and the top slice of the z-stack side by side
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
ax[0].imshow(F_func[0], cmap='gray')
ax[0].set_title("Movie")
# ax[1].imshow(fov_image, cmap='gray')
# ax[1].set_title("FOV")
ax[1].imshow(Zstack[:, :, -1], cmap='gray')
ax[1].set_title("Z-stack")
plt.show()

In [ ]:
# Save F_func as tiff
f_func_filepath = export_path / f"F_func_T_{Nframe}_Y_{Ny}_X_{Nx}.tiff"
with TiffWriter(f_func_filepath, bigtiff=True) as tif:
    tif.write(F_func.astype(np.uint16), photometric='minisblack')

In [ ]:
def make_f_anat(Zstack, valid_indices, frame_xshift = None, frame_yshift = None):
    """
    Apply affine transformations to frames in Zstack based on shifts, and return the modified stack.
    """        
    Z_frame_shifted = np.zeros((Zstack.shape[0], Zstack.shape[1], len(valid_indices)), dtype=np.float32)
    
    if frame_xshift is None:
        frame_xshift = np.zeros(Zstack.shape[2])
    if frame_yshift is None:
        frame_yshift = np.zeros(Zstack.shape[2])
            
    for stack_idx, z_idx in enumerate(valid_indices):
        transformation_matrix = np.array([[1, 0, -frame_yshift[z_idx]], [0, 1, -frame_xshift[z_idx]]])
        Z_frame_shifted[:, :, stack_idx] = affine_transform(
            Zstack[:, :, z_idx], transformation_matrix, order=1, mode='constant', cval=0.0, prefilter=False)

        # from skimage.transform import warp, AffineTransform
        # transform = AffineTransform(translation=(-frame_yshift[z_idx], -frame_xshift[z_idx]))
        # Z_frame_shifted[:, :, stack_idx] = warp(Zstack[:, :, z_idx], transform.inverse, order=1, mode='constant', cval=0.0)

    # Clip Z_frame_shifted to uint16 range
    return np.clip(Z_frame_shifted, 0, 2**16-1)

if os.path.exists(mesmerize_path / "F_anat_rigid_T_{Nframe}_Y_{Ny}_X_{Nx}.tiff"):
    # Load F_anat from tiff
    with TiffFile(mesmerize_path / "F_anat_rigid_T_{Nframe}_Y_{Ny}_X_{Nx}.tiff") as tif:
        F_anat = tif.asarray()
else:
    # Create F_anat with the shifted z-stack frames
    F_anat_rigid = [make_f_anat(Zstack, 
                        [index for index in [zpos[frameNum]] if 0 <= index < Zstack.shape[2]]).squeeze()
                        # xshift[:, frameNum], yshift[:, frameNum]
                        for frameNum in range(Nframe)]

    F_anat_rigid = np.array(F_anat_rigid)
    # Save F_anat as tiff
    f_anat_filepath = export_path / f"F_anat_rigid_T_{Nframe}_Y_{Ny}_X_{Nx}.tiff"
    with TiffWriter(f_anat_filepath, bigtiff=True) as tif:
        tif.write(F_anat_rigid.astype(np.uint16), photometric='minisblack')

**Standard method to compute F_anat**

In [ ]:

# Translate zstack frames to minimize shift with mean of registered tseries frames
zpos = z_correlation['zpos']
zpos_0 = mode(zpos).mode
Zstack_0 = Zstack[:, :, zpos_0]

# Perform image registration
fov_image = np.mean(F_func, axis=0)
shift, _ , _ = phase_cross_correlation(Zstack_0, fov_image, upsample_factor=10)

# # Create the transformation matrix (for translation only)
tform = np.eye(3)
tform[0, 2] = -shift[1]
tform[1, 2] = -shift[0]

# Translate z-stack frames to minimize shift with mean of registered tseries frames (FOV)
Zstack_reg = np.zeros_like(Zstack)
for iz in range(Nz):
    Zstack_reg[:, :, iz] = warp(np.float32(Zstack[:, :, iz]), tform)
# Make a movie of the z-stack frames that follows zcorr (movement in depth)
F_anat = Zstack_reg[:, :, zpos]
# Re-order to (T) x Y x X
F_anat = np.moveaxis(F_anat, -1, 0)
# Clip to uint16
F_anat = np.clip(F_anat, 0, 2**16-1)

In [ ]:
# Save F_anat as tiff
f_anat_classic_filepath = export_path / f"F_anat_rigid_classic_T_{Nframe}_Y_{Ny}_X_{Nx}.tiff"
with TiffWriter(f_anat_classic_filepath, bigtiff=True) as tif:
    tif.write(F_anat.astype(np.uint16), photometric='minisblack')

In [ ]:
# Plot the first image of 
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
ax[0].imshow(F_anat[0], cmap='gray')
ax[0].set_title("F_anat")
ax[1].imshow(Zstack[:, :, z_correlation['zpos'][0]], cmap='gray')
# ax[1].imshow(Zstack_0, cmap='gray')
ax[1].set_title("Z-stack")
plt.show()

**Compute correlation between F_func and F_anat_rigid**

In [ ]:
F_func_flattened = F_func.ravel()
F_anat_rigid_flattened = F_anat_rigid.ravel()
correlations_f_anat_rigid = np.corrcoef(F_func_flattened, F_anat_rigid_flattened)[0, 1]
print(f"Correlation between F_func and F_anat_rigid: {correlations_f_anat_rigid}")

# Save the z-correlation file
zcorr_filepath = export_path / "correlation_f_func_f_anat_rigid.npy"
np.save(zcorr_filepath, correlations_f_anat_rigid)

**Compute correlation between F_func and F_anat**

In [ ]:
F_func_flattened = F_func.ravel()
F_anat_flattened = F_anat.ravel()
correlations_f_anat = np.corrcoef(F_func_flattened, F_anat_flattened)[0, 1]
print(f"Correlation between F_func and F_anat: {correlations_f_anat}")

# Save the z-correlation file
zcorr_filepath = export_path / "correlation_f_func_f_anat.npy"
np.save(zcorr_filepath, correlations_f_anat)

### Generate non-rigid F_anat
We divide the frames in patches and find the linear (least-squares) regression between the F_func movie and the z-stack for each patch at several depths around zpos.   
* Function `patch_regress` from `compute_zcorr`,
* Select the z indices to use in the z-stack (i.e., +/- 5 steps around zpos = sub-Z-stack). 
* Use the `mcorr` parameters to define patch size based on strides (steps) and overlap. 
* For each frame:  
  * Shift the sub-Z-stack using the x/y shift computed earlier.  
  * Loop through each patch  
    * For each step in the sub-Z-stack, compute the linear regression with the corresponding motion-corrected movie patch.  
    * Get preferred depth of patch = index of the max of smooth $R^2$  
    * Append linear regression results at that depth to patch_regress_results  
  * Returns patch_correlations = patch regress_results for all frames  
* Compute zones, which correspond to patch overlaps  
  * Returns labeled_zones (zones with sequential numbers column-order) and zone_pattern (zones with non-sequential numbers)
* Compute the composite (non-rigid) F_anat
  * For each frame 
    * For each zone, average the zone across preferred patch depths
    * Stitch the zones 
  * Returns F_anat_non_rigid, the depth-motion corrected F_anat movie

**Compute patch correlations**

In [ ]:
def patch_regress(frame_data, shift_frames=False):
    """
    Calculate the linear least-squares regression between the F_func movie and the z-stack
    for each patch at several depths around zpos.
    
    Inputs: 
    - frame_data: tuple containing the frame number, the frame data, the z-stack, the x/y shifts, the dimensions of the movie, the patch size, the step size, the current zpos, and the valid indices.
    
    Outputs:
    - results: list of dictionaries containing the r_squared, slope, intercept, frame number, patch number, patch x/y limits, patch z index, and patch z position. 
    """
    
    # np.seterr(over='raise')
        
    # fram_start_time = time.time()

    frameNum, frame_F_func, Zstack, frame_xshift, frame_yshift, Nx, Ny, patch_size, step_size, current_zpos, valid_indices = frame_data
        
    Z_frame_shifted = np.zeros_like(Zstack)
    patch_regress_results = []

    if shift_frames:
        for z_idx in valid_indices:
            frameXshift = -frame_xshift[z_idx]
            frameYshift = -frame_yshift[z_idx]
            transformation_matrix = np.array([[1, 0, frameYshift], [0, 1, frameXshift]])
            Z_frame_shifted[:, :, z_idx] = affine_transform(Zstack[:, :, z_idx], transformation_matrix, order=1, mode='constant', cval=0.0, prefilter=False)
    else:
        Z_frame_shifted = Zstack
    # Clip Z_frame_shifted to uint16 range
    Z_frame_shifted = np.clip(Z_frame_shifted, 0, 2**16-1)
        
    # Initialize patch number
    patch_num = -1
    
    # Loop over patches, moving vertically (along columns) first, then horizontally (along rows)    
    for i in range(0, Nx, step_size[1]):
        for j in range(0, Ny, step_size[0]):
            # Allow for a last partial patch on the right and bottom borders, but not more
            if i + patch_size[1] > Nx and patch_size_x < patch_size[1] and patch_size_y < patch_size[0]:
                continue 
            if j + patch_size[0] > Ny and patch_size_y < patch_size[0]:
                continue 
            # Adjust the patch size for patches on the bottom and right borders
            patch_size_x = patch_size[1] if i + patch_size[1] <= Nx else Nx - i
            patch_size_y = patch_size[0] if j + patch_size[0] <= Ny else Ny - j

            # Get the patch from the frame data
            T_patch = frame_F_func[j:j+patch_size_y, i:i+patch_size_x, ].ravel()
            # Increment patch number
            patch_num += 1
            
            # print(f"Frame {frameNum}, Patch {patch_num}, Patch Size: {patch_size_y}x{patch_size_x}")
            
            # Initialize results list
            patch_results = []
            # Loop over valid depth indices
            for z_idx in valid_indices:
                Z_patch = Z_frame_shifted[j:j+patch_size_y, i:i+patch_size_x, z_idx].ravel()
                if T_patch.size == Z_patch.size:
                    _, _, r_value, _, _ = linregress(T_patch, Z_patch)
                    # try:
                    patch_results.append({
                    'r_squared': r_value**2, 'Z_patch': Z_patch, 
                    'frame_num': frameNum, 'patch_number': patch_num,
                    'patch_x_lims': [i, i+patch_size_x], 'patch_y_lims': [j, j+patch_size_y],
                    'patch_z_idx': int(z_idx) - int(current_zpos), 'patch_z_pos': z_idx,
                    'patch_x_shift': frame_xshift[z_idx], 'patch_y_shift': frame_yshift[z_idx]
                    })
                    # not needed: 'slope': slope, 'intercept': intercept,
                    
                    # except FloatingPointError:
                    #     print("Overflow Error")
                    #     # print(f"z_idx: {z_idx}, current_zpos: {current_zpos}")
                        
            # Smooth R^2 values over valid depth indices
            r_squares = [result['r_squared'] for result in patch_results]
            r_squares = gaussian_filter(r_squares, sigma=1)
            # Get the max R^2 value's index
            max_r_square_idx = np.argmax(r_squares)
            # Append the result corresponding to the max R^2 value to patch_regress_results
            patch_regress_results.append(patch_results[max_r_square_idx])                      
                        
    # frame_end_time = time.time()
    # print(f"Frame {frameNum} processed in {frame_end_time - fram_start_time:.2f} seconds")
            
    return patch_regress_results

In [ ]:
# Define patch size based on Caiman's parameters. Width of patch = strides+overlaps.
mcorr_params = parameters['params_mcorr']['main']
step_size = mcorr_params['strides'] 
patch_overlap = mcorr_params['overlaps']
# Calculate patch size for both x and y dimensions 
patch_size = [step + overlap for step, overlap in zip(step_size, patch_overlap)]
print(f"Patch size: {patch_size}, Step size: {step_size}, Patch overlap: {patch_overlap}")

In [ ]:
# if patch_correlations.parquet exists, load it 

# if (export_path / 'patch_correlations_15zplanes.parquet').exists():
#     patch_correlations_df = pd.read_parquet(export_path / 'patch_correlations_15zplanes.parquet')
if (export_path / 'patch_correlations.parquet').exists():
    patch_correlations_df = pd.read_parquet(export_path / 'patch_correlations.parquet')   
    patch_correlations_df['Z_patch'] = patch_correlations_df['Z_patch'].apply(lambda x: np.frombuffer(x, dtype=np.uint16))     
else: 
    start_time = time.time()
    # Prepare data for parallel processing
    frame_data_list = []
    for frameNum in range(Nframe):
        current_zpos = zpos[frameNum]
        # The number of z-stack frames to consider is 11 (+/- 5 frames around the current zpos)
        indices = [current_zpos + i for i in range(-5, 6)]
        # Check for valid z indices
        valid_indices = [index for index in indices if 0 <= index < Zstack.shape[2]]
        # Append the frame data to the list
        # If applying shifts to the frames, set shift_frames=True
        # frame_data_list.append((frameNum, F_func[frameNum, :, :],
        #                         Zstack, xshift[:, frameNum], yshift[:, frameNum],
        #                         Nx, Ny, patch_size, step_size,
        #                         zpos[frameNum], valid_indices))
        frame_data_list.append((frameNum, F_func[frameNum, :, :],
                        Zstack, np.zeros(Zstack.shape[2]), np.zeros(Zstack.shape[2]),
                        Nx, Ny, patch_size, step_size,
                        zpos[frameNum], valid_indices))
        
    # Use Parallel to parallelize the task 
    patch_regress_results = Parallel(n_jobs=-1)(delayed(patch_regress)(frame_data) for frame_data in tqdm(frame_data_list, desc="Find the correlation between patches in the F_func movie and the anat z-stack over frames"))

    # Flatten list of lists returned by map
    patch_correlations = [item for sublist in patch_regress_results for item in sublist]

    # Convert to DataFrame
    patch_correlations_df = pd.DataFrame(patch_correlations)
    
    # Save patch correlations to a Parquet file 
    patch_correlations_filepath = export_path / 'patch_correlations.parquet' 
    # # Saving / loading the Z_patch column is tricky, as its quite large. 
    # # One option is to convert it to binary before saving
    # patch_correlations_df['Z_patch'] = patch_correlations_df['Z_patch'].apply(lambda x: x.tobytes())  
    # # Another option would be to save it separately.
    # patch_correlations_df.to_parquet(patch_correlations_filepath, index=False)
    
    # Display execution time in mn and seconds
    exect_time = time.time() - start_time

    print("Patch correlation computation took: ", exect_time // 60, "mn and ", exect_time % 60, "seconds")
    
# For 1600 frames, correlation computation took:  2.0 mn and  36.99588894844055 seconds
# "11 layer" patch regress: 6.0 mn

In [ ]:
def calculate_zones(patch_correlations_df, Ny, Nx):
    """
    Find zones (i.e., how patches overlap) in the patch_correlations_df DataFrame.
    """
    start_time = time.time()
    
    # Initialize a mask for zone calculation
    mask = np.zeros((Ny, Nx), dtype=int)

    # Use the first frame's patches to define zones
    first_frame_patches = patch_correlations_df[patch_correlations_df['frame_num'] == patch_correlations_df['frame_num'].min()]
    for index, row in first_frame_patches.iterrows():
        x_start, x_end = row['patch_x_lims']
        y_start, y_end = row['patch_y_lims']
        mask[y_start:y_end, x_start:x_end] += 1
    
    # Label zones based on overlapping patches. Assign a unique number to each zone

    # Initialize the zone_pattern array to store zone identifiers
    zone_pattern = np.zeros((Ny, Nx), dtype=int)

    # Change the mask values from 4 to 3 (even to odd)
    mask = np.where(mask == 4, 3, mask)
    # Process for odd values (mask values that are odd become 1, others 0)
    binary_mask_odd = (mask % 2 == 1).astype(int)
    labeled_zones_odd, num_zones_odd = label(binary_mask_odd)

    # Assign unique labels to zone_pattern for odd zones
    zone_id_offset = 0  # Start zone IDs from 1 for odd zones
    for i in range(1, num_zones_odd + 1):
        zone_pattern[labeled_zones_odd == i] = zone_id_offset + i

    # Update the offset for even zones to continue unique labeling
    zone_id_offset += num_zones_odd

    # Process for even values (mask values that are even and non-zero become 1, others 0)
    binary_mask_even = ((mask % 2 == 0) & (mask != 0)).astype(int)
    labeled_zones_even, num_zones_even = label(binary_mask_even)

    # Assign unique labels to zone_pattern for even zones
    for i in range(1, num_zones_even + 1):
        zone_pattern[labeled_zones_even == i] = zone_id_offset + i

    # Flatten the array column-wise
    zone_pattern_col_flat = zone_pattern.flatten(order='F')

    # Get the unique zones and their unique indices
    _, inverse_indices = np.unique(zone_pattern_col_flat, return_inverse=True)
    unique_indices = pd.unique(inverse_indices) + 1
    
    # Re-label zones to have continuous IDs
    zone_pattern_contig = np.zeros_like(zone_pattern, dtype=np.uint16)
    for new_id, zone_id in enumerate(unique_indices):
        zone_pattern_contig[zone_pattern == zone_id] = new_id
        
    # Create a look-up table. For each pixel coordinates, get the zone ID from zone_pattern_contig
    # Convert numpy uint16 to Python int during dictionary creation
    # lookup_table = {index: int(value) for index, value in np.ndenumerate(zone_pattern_contig)}
    
    print(f"Patches divided into zones of overlap in {time.time() - start_time:.2f} seconds")
    
    # # Plot the zone pattern, no tick or label, with one segmentation color for each different pixel value.
    # plt.imshow(zone_pattern, cmap='plasma')
    # plt.xticks([])
    # plt.yticks([])
    # # plt.show()
    # # Save to export path / plots
    # plot_export_path = export_path / 'plots'
    # plot_export_path.mkdir(parents=True, exist_ok=True)
    # plt.savefig(plot_export_path / 'zone_pattern.png')
    
    return zone_pattern_contig, zone_pattern

In [ ]:
labeled_zones, zone_pattern= calculate_zones(patch_correlations_df, Ny, Nx)

In [ ]:
print("Saving labeled_zones")
labeled_zones_df = pd.DataFrame(labeled_zones)
labeled_zones_filepath = export_path / 'labeled_zones.csv'
labeled_zones_df.to_csv(labeled_zones_filepath, index=False, header=False)

In [ ]:
def make_composite_f_anat(patch_correlations_df, labeled_zones):
    fields = ['r_squared', 'patch_number', 'patch_z_pos', 'patch_x_shift', 'patch_y_shift', 'Z_patch']
    # start_time = time.time()
    zone_data = []
    # F_anat_non_rigid = np.zeros(np.unique(patch_correlations_df['frame_num']), labeled_zones.shape)
    F_anat_non_rigid = np.zeros((len(np.unique(patch_correlations_df['frame_num'])), *labeled_zones.shape))

    # Iterate over frames
    for frame_idx, (frame_num, group) in enumerate(patch_correlations_df.groupby('frame_num')):        
        # sums = {field: np.zeros((np.max(labeled_zones)+1,)) for field in fields}
        sums = {field: np.zeros((np.max(labeled_zones)+1,)) for field in fields if field != 'Z_patch'}
        sums['Z_patch'] = {zone_id: [] for zone_id in np.unique(labeled_zones)}

        count = np.zeros((np.max(labeled_zones)+1,))
        
        composite_f_anat_frame = np.zeros(labeled_zones.shape)

        # Populate sums and counts per zone                
        for _, row in group.iterrows():
            x_start, x_end = row['patch_x_lims']
            y_start, y_end = row['patch_y_lims']
            
            # Extract the patch area from the labeled_zones
            patch_2D = row['Z_patch'].reshape(y_end - y_start, x_end - x_start)
            
            patch_zone_labels = labeled_zones[y_start:y_end, x_start:x_end]
            zone_ids = np.unique(patch_zone_labels)

            for zone_id in zone_ids:
                # Find coordinates of the current zone_id within the patch
                zone_mask = (patch_zone_labels == zone_id)
                # Get bounding box of the zone for 2d zone_specific_patch
                rows, cols = np.where(zone_mask)
                min_row, max_row = rows.min(), rows.max()
                min_col, max_col = cols.min(), cols.max()
                for field in fields:
                    if field == 'Z_patch':
                        # Extract the part of Z_patch that corresponds to the current zone_id.
                        # This returns a 1D array, ordered row-wise.
                        # zone_specific_patch = patch_2D[zone_mask]
                        #  in 2D for sanity check
                        zone_specific_patch = patch_2D[min_row:max_row+1, min_col:max_col+1]
                        sums[field][zone_id].append(zone_specific_patch)
                    else:
                        sums[field][zone_id] += row[field]
                count[zone_id] += 1

        # Calculate averages for each zone            
        averages = {field: np.zeros((np.max(labeled_zones)+1,)) for field in fields if field != 'Z_patch'}
        averages['Z_patch'] = {}

        for field in sums:
            if field == 'Z_patch':
                for zone_id, arrays in sums[field].items():
                    try:
                        if arrays:
                            averages[field][zone_id] = np.mean(arrays, axis=0)
                    except Exception as e:
                        # print(e)
                        print(f"Frame {frame_num}, Zone {zone_id}")
            else:
                averages[field] = np.divide(sums[field], count, where=(count > 0))

        # Collect data per zone
        unique_zones = np.unique(labeled_zones)
        for zone_id in unique_zones:
            zone_info = {'frame_num': frame_num, 'zone_id': zone_id}
            if count[zone_id] > 0:
                y_indices, x_indices = np.where(labeled_zones == zone_id)
                y_min, y_max = y_indices.min(), y_indices.max() + 1
                x_min, x_max = x_indices.min(), x_indices.max() + 1
                for field in fields[:-1]:
                    zone_info[field] = averages[field][zone_id]
                zone_data.append(zone_info)
                # reshape Zpatch to 2D and insert into composite_f_anat_frame               
                if 'Z_patch' in averages and zone_id in averages['Z_patch']:
                    # zone_2D = averages['Z_patch'][zone_id].reshape((y_max - y_min, x_max - x_min))
                    composite_f_anat_frame[y_min:y_max, x_min:x_max] = averages['Z_patch'][zone_id]
        
        F_anat_non_rigid[frame_idx, :, :] = composite_f_anat_frame
        # %matplotlib inline
        # plot the composite_f_anat_frame
        # plt.imshow(F_anat_non_rigid[frame_idx, :, :], cmap='gray')
        # plt.title(f"Composite F_anat for frame {frame_num}")
        # plt.show()
        # # save the composite_f_anat_frame
        # composite_f_anat_frame_path = Path(f"/home/wanglab/data/2P/Scnn1aAi14_B1M0/04222024/mesmerize_zcorrel/plots/composite_f_anat_frame_{frame_num}.tiff")
        # plt.imsave(composite_f_anat_frame_path, F_anat_non_rigid[frame_idx, :, :], cmap='gray')
        
    zone_df = pd.DataFrame(zone_data)
    
    # print(f"Zone data computed in {time.time() - start_time:.2f} seconds")
    
    return zone_df, F_anat_non_rigid

In [ ]:
# Iterate over zones. For pixels with overlap between patches, average the correlation values.
# zone_df = make_composite_f_anat(patch_correlations_df, labeled_zones)
# zone_df = pd.concat(Parallel(n_jobs=-1)(delayed(make_composite_f_anat)(group, labeled_zones) for frame_num, group in tqdm(patch_correlations_df.groupby('frame_num'), desc="Averaging correlations for each zone")))
    
# Using ProcessPoolExecutor to parallelize
groups = list(patch_correlations_df.groupby('frame_num'))
with ProcessPoolExecutor() as executor:
    # Submit frames to make_composite_f_anat
    futures = {executor.submit(make_composite_f_anat, group, labeled_zones): frame_num for frame_num, group in groups}

    # Collect results as they complete
    results = []
    frame_arrays = []
    frame_nums = [] 
    for future in tqdm(as_completed(futures), total=len(futures), desc="Generating composite F_anat"):
        frame_num = futures[future]  # Retrieve frame number from the dictionary
        zone_data, image_data = future.result()
        results.append(zone_data)
        frame_arrays.append(image_data)
        frame_nums.append(frame_num) 
        
# Concatenate all DataFrame results
zone_df = pd.concat(results)

# F_anat_non_rigid = np.concatenate(frame_arrays, axis=0)
# Just in case the frames are not in order, sort frames and arrays according to the frame number
sorted_indices = np.argsort(frame_nums)
F_anat_non_rigid = np.concatenate([frame_arrays[idx] for idx in sorted_indices], axis=0)
# Clip to uint16 range
F_anat_non_rigid = np.clip(F_anat_non_rigid, 0, 2**16-1).astype(np.float32)

In [ ]:
# Save F_anat_non_rigid
composite_f_anat_filepath = export_path / f"F_anat_non_rigid_T_{Nframe}_Y_{Ny}_X_{Nx}.tiff"
# Save movie to a tiff file
with TiffWriter(composite_f_anat_filepath, bigtiff=True) as tif:
    tif.write(F_anat_non_rigid.astype(np.uint16), photometric='minisblack') 

<div class="alert alert-info">
    <h3 style="margin-top: 0;">Compute a fit between F_anat_non_rigid and F_func for rigid motion correction.</h3>  

Load F_func (motion-corected movie). 

Compute F_anat_rigid: load Zstack, then find Zstack[:, :, zpos(frame)] using corresponding xshifts/yshifts.  

Linearize the two 3D arrays and find correlation and fit (b).  
</div>

In [ ]:
### Compute correlation between F_func and F_anat_non_rigid 

In [ ]:
F_func_flattened = F_func.ravel()
F_anat_non_rigid_flattened = F_anat_non_rigid.ravel()
correlations_f_anat_non_rigid = np.corrcoef(F_func_flattened, F_anat_non_rigid_flattened)[0, 1]
print(f"Correlation between F_func and F_anat_non_rigid: {correlations_f_anat_non_rigid}")

# Save the z-correlation file
zcorr_filepath = export_path / "correlation_f_func_f_anat_non_rigid.npy"
np.save(zcorr_filepath, correlations_f_anat_non_rigid)

In [ ]:
# Concatenate F_anat_rigid, F_anat_non_rigid and F_func
F_concat = np.concatenate([F_anat_rigid, F_anat_non_rigid, F_func], axis=2)
# Save F_concat as tiff
f_concat_filepath = export_path / f"Concat_F_anat_rigid_F_anat_non_rigid_F_func.tiff"
with TiffWriter(f_concat_filepath, bigtiff=True) as tif:
    tif.write(F_concat.astype(np.uint16), photometric='minisblack')
    

In [ ]:
# Print F_func and F_anat_non_rigid shapes, data type and min/max range
print(f"F_func shape: {F_func.shape}, data type: {F_func.dtype}, min: {F_func.min()}, max: {F_func.max()}")
print(f"F_anat_non_rigid shape: {F_anat_non_rigid.shape}, data type: {F_anat_non_rigid.dtype}, min: {F_anat_non_rigid.min()}, max: {F_anat_non_rigid.max()}")

In [ ]:
# Compute fit between F_func and F_anat_non_rigid using linear regression
F_func_flattened = F_func.ravel()
F_anat_non_rigid_flattened = F_anat_non_rigid.ravel()
#  Use linregress to model F_func = slope * F_anat_non_rigid + intercept.
slope, intercept, r_value, p_value, std_err = linregress(F_anat_non_rigid_flattened, F_func_flattened)

print(f"R^2 value between F_func and F_anat_non_rigid: {r_value**2}")
print(f"Slope: {slope}, Intercept: {intercept}")

In [ ]:
# Rescale and subtract the movement-induced changes from F_func
F_anat_non_rigid_adjusted = slope * F_anat_non_rigid + intercept
F_corrected = F_func - F_anat_non_rigid_adjusted
print(f"F_corrected shape: {F_corrected.shape}, data type: {F_corrected.dtype}, min: {F_corrected.min()}, max: {F_corrected.max()}")
# Clip F_corrected to uint16 range
F_corrected = np.clip(F_corrected, 0, 2**16-1)
# Save to tiff 
f_corrected_filepath = export_path / f"F_func_corrected_T_{Nframe}_Y_{Ny}_X_{Nx}.tiff"
with TiffWriter(f_corrected_filepath, bigtiff=True) as tif:
    tif.write(F_corrected.astype(np.uint16), photometric='minisblack')